In [0]:

# ============================================================
# CELDA 1. EDA GOLD: ESTACIONALIDAD Y HORARIOS
# ============================================================

# Analizamos cómo cambia el porcentaje de vuelos
# retrasados según el mes, día de la semana y hora.
#
# Consideramos retraso cuando la salida ocurre
# 15 minutos o más después de lo programado.
#
# Fuente: tabla Gold de vuelos.

from pyspark.sql import functions as F
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# 1. CARGAR LOS DATOS GOLD
# ------------------------------------------------------------

vuelos = spark.table(
    "workspace.gold_flights.flights_features"
)

# Solo descargaremos resultados agregados para graficar.
# No convertimos los 8.6 millones de vuelos a pandas.

# ------------------------------------------------------------
# 2. PORCENTAJE DE RETRASOS POR MES
# ------------------------------------------------------------

por_mes = (
    vuelos
    .groupBy("month")
    .agg(
        F.count("*").alias("total_flights"),
        F.round(
            F.avg("delayed_15") * 100, 2
        ).alias("delay_rate")
    )
    .orderBy("month")
    .toPandas()
)

meses = [
    "Ene", "Feb", "Mar", "Abr", "May", "Jun",
    "Jul", "Ago", "Sep", "Oct", "Nov", "Dic"
]

por_mes["month_name"] = por_mes["month"].apply(
    lambda x: meses[int(x) - 1]
)

plt.figure(figsize=(11, 5))
plt.bar(por_mes["month_name"], por_mes["delay_rate"])
plt.title("Porcentaje de vuelos retrasados por mes")
plt.xlabel("Mes")
plt.ylabel("Vuelos con retraso ≥15 min (%)")
plt.grid(axis="y", alpha=0.2)
plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# 3. PORCENTAJE DE RETRASOS POR DÍA DE LA SEMANA
# ------------------------------------------------------------

# En Spark, 1 = domingo y 7 = sábado.

por_dia = (
    vuelos
    .groupBy("day_of_week")
    .agg(
        F.count("*").alias("total_flights"),
        F.round(
            F.avg("delayed_15") * 100, 2
        ).alias("delay_rate")
    )
    .orderBy("day_of_week")
    .toPandas()
)

nombres_dias = {
    1: "Dom",
    2: "Lun",
    3: "Mar",
    4: "Mié",
    5: "Jue",
    6: "Vie",
    7: "Sáb"
}

por_dia["day_name"] = por_dia["day_of_week"].map(
    nombres_dias
)

plt.figure(figsize=(10, 5))
plt.bar(por_dia["day_name"], por_dia["delay_rate"])
plt.title("Porcentaje de vuelos retrasados por día de la semana")
plt.xlabel("Día")
plt.ylabel("Vuelos con retraso ≥15 min (%)")
plt.grid(axis="y", alpha=0.2)
plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# 4. PORCENTAJE DE RETRASOS POR HORA PROGRAMADA
# ------------------------------------------------------------

por_hora = (
    vuelos
    .groupBy("departure_hour")
    .agg(
        F.count("*").alias("total_flights"),
        F.round(
            F.avg("delayed_15") * 100, 2
        ).alias("delay_rate")
    )
    .orderBy("departure_hour")
    .toPandas()
)

plt.figure(figsize=(11, 5))
plt.plot(
    por_hora["departure_hour"],
    por_hora["delay_rate"],
    marker="o"
)
plt.title("Porcentaje de vuelos retrasados por hora de salida")
plt.xlabel("Hora programada de salida (hora local)")
plt.ylabel("Vuelos con retraso ≥15 min (%)")
plt.xticks(range(0, 24, 2))
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# 5. MOSTRAR LOS RESULTADOS PRINCIPALES
# ------------------------------------------------------------

print("=" * 55)
print("EDA GOLD: ESTACIONALIDAD Y HORARIOS")
print("=" * 55)

print("\nMES CON MAYOR PORCENTAJE DE RETRASOS")
display(
    por_mes.sort_values(
        "delay_rate", ascending=False
    ).head(3)
)

print("\nDÍAS CON MAYOR PORCENTAJE DE RETRASOS")
display(
    por_dia.sort_values(
        "delay_rate", ascending=False
    ).head(3)
)

print("\nHORAS CON MAYOR PORCENTAJE DE RETRASOS")
display(
    por_hora.sort_values(
        "delay_rate", ascending=False
    ).head(5)
)

print("\nEDA GOLD COMPLETADO.")


In [0]:

# ============================================================
# CELDA 2. VALIDACIÓN DE RETRASOS POR HORA
# ============================================================

# Revisamos cuántos vuelos tenemos por hora para
# evitar conclusiones basadas en muestras pequeñas.

from pyspark.sql import functions as F

vuelos = spark.table(
    "workspace.gold_flights.flights_features"
)

revision_horas = (
    vuelos
    .groupBy("departure_hour")
    .agg(
        F.count("*").alias("total_vuelos"),
        F.sum("delayed_15").alias("vuelos_retrasados"),
        F.round(
            F.avg("delayed_15") * 100, 2
        ).alias("porcentaje_retrasos")
    )
    .orderBy("departure_hour")
)

print("VALIDACIÓN DE HORARIOS DE MADRUGADA")

display(
    revision_horas.filter(
        F.col("departure_hour").between(0, 6)
    )
)


In [0]:

# ============================================================
# CELDA 3. GRÁFICA HORARIA CON TAMAÑO DE MUESTRA
# ============================================================

# Para interpretar correctamente los porcentajes,
# distinguimos las horas con suficientes vuelos.
# No eliminamos registros de nuestra tabla Gold.

import matplotlib.pyplot as plt

datos_horas = revision_horas.toPandas()

# Umbral descriptivo para mostrar la tendencia.
# No es un requisito estadístico universal.
MIN_VUELOS = 100

horas_principales = datos_horas[
    datos_horas["total_vuelos"] >= MIN_VUELOS
]

horas_limitadas = datos_horas[
    datos_horas["total_vuelos"] < MIN_VUELOS
]

plt.figure(figsize=(11, 5))

plt.plot(
    horas_principales["departure_hour"],
    horas_principales["porcentaje_retrasos"],
    marker="o",
    label="Horarios con 100+ vuelos"
)

plt.scatter(
    horas_limitadas["departure_hour"],
    horas_limitadas["porcentaje_retrasos"],
    marker="x",
    s=90,
    label="Muestra menor a 100 vuelos"
)

plt.title("Retrasos por hora de salida (con tamaño de muestra)")
plt.xlabel("Hora programada de salida (hora local)")
plt.ylabel("Vuelos retrasados ≥15 min (%)")
plt.xticks(range(0, 24))
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()
plt.show()

print("HORARIOS CON MUESTRA LIMITADA")
display(horas_limitadas)
